In [ ]:
# Locate the repository when opened from any experiment subdirectory.
import os
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next((p for p in (_here, *_here.parents)
             if (p / "smoke_test.py").is_file()
             and (p / "configs/config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the PGAE-tau repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"Repository: {ROOT}")


# Anomaly Transformer: FDIA Detection / FDI 攻击检测

**Author**: Hawkyu | **Date**: 2026-06-20

Evaluates **Anomaly Transformer** (Xu et al., ICLR 2022). Candidate thresholds come from validation-score quantiles. Selection uses normal-test FPR: choose the candidate closest to 0.05 from below, or the smallest available FPR when all candidates exceed the target.
候选阈值来自验证集分数的分位数。按正常测试集 FPR 选择最接近且不超过 0.05 的候选值；若全部超过目标，则选择 FPR 最小的候选值。

The anomaly score combines association discrepancy between prior and series associations with reconstruction error.
Anomaly score = softmax(-discrepancy) ⊙ reconstruction_error.

In [ ]:
import sys
sys.path.append('.')
import matplotlib.pyplot as plt
from configs.config import sys_config
from configs.nn_setting import nn_setting
from tqdm import tqdm
from matplotlib import cm
import torch, numpy as np
from data.loaders import load_case, load_measurement, load_load_pv, load_dataset
from models.baselines.anomaly_transformer import AnomTrans
from data.dataset import scaler

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
TARGET_FPR = 0.05

class Evaluation:
    """Evaluate Anomaly Transformer with quantile-based threshold selection."""
    def __init__(self, case_class):
        self.case_class = case_class
        self.case_class.gen_torch_constant()
        


        # Load the model and trained parameters
        model = AnomTrans().to(nn_setting['device'])
        model_path = f"saved_model/{sys_config['case_name']}/AnomTrans.pt"
        
        model.load_state_dict(torch.load(model_path, map_location=torch.device(nn_setting['device']),  weights_only=True))
        model.eval()
        self.model = model
        self.device = nn_setting['device']
    
    def anomaly_score(self, x):
        """Compute anomaly score for a batch."""
        if not isinstance(x, torch.Tensor):
            x = torch.from_numpy(x).float()
        x = x.to(self.device)
        if x.dim() == 2:
            x = x.unsqueeze(0)
        return self.model.anomaly_score(x).cpu().item()
    
    def find_threshold(self, valid_dl_s, test_dl_s):
        """Select a validation-quantile threshold using normal-test FPR."""
        valid_scores = []
        for inp, out in valid_dl_s:
            valid_scores.append(self.anomaly_score(inp))
        valid_scores = np.array(valid_scores)
        
        test_scores = []
        for inp, out in test_dl_s:
            test_scores.append(self.anomaly_score(inp))
        test_scores = np.array(test_scores)
        
        # Dense quantile search
        dense_q = np.unique(np.sort(np.concatenate([
            np.linspace(0.80, 0.90, 21), np.linspace(0.91, 0.99, 17),
            [0.991, 0.993, 0.995, 0.997, 0.999],
        ])))
        thresholds = np.quantile(valid_scores, dense_q)
        
        FP_counts = np.array([np.sum(test_scores >= thr) for thr in thresholds])
        FPRs = FP_counts / len(test_scores)
        vi = np.where(FPRs <= TARGET_FPR)[0]
        best_idx = vi[np.argmax(FPRs[vi])] if len(vi) > 0 else np.argmin(FPRs)
        self.best_thr = thresholds[best_idx]
        self.best_fpr = FPRs[best_idx]
        self.FP_DDD = (test_scores >= self.best_thr).tolist()
        self.FP_count = int(np.sum(self.FP_DDD))
        self.TN_count = len(self.FP_DDD) - self.FP_count
        self.FPR_DD_summary = FPRs
        self.thresholds_list = thresholds
        
        print(f'Threshold: {self.best_thr:.6f} | FPR={self.best_fpr:.4f}')
        print(f'Normal samples: FP={self.FP_count}, TN={self.TN_count}')
        return self.best_thr, self.best_fpr

In [ ]:
# ---- Attack Strength Sweep ----
ANG_STRS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
ANG_NOS = [1, 2, 3]

def sweep_attack_strength(detector, test_dl_u, scaler_):
    """Evaluate attack strengths at the selected threshold."""
    strength_results = []
    for ang_str in ANG_STRS:
        all_tpr = []
        for ang_no in ANG_NOS:
            for _, (idx, inp, vp, vl) in tqdm(enumerate(test_dl_u),
                                               desc=f'  ang_str={ang_str:.2f} no={ang_no}',
                                               total=len(test_dl_u), leave=False):
                vl = vl.flatten()
                z_att, v_att = detector.case_class.gen_fdi_att_dd(
                    z_noise=inp, v_est_last=vl,
                    ang_no=ang_no, mag_no=0, ang_str=ang_str, mag_str=0)
                z_s = scaler_(z_att.numpy() if hasattr(z_att, 'numpy') else z_att)
                score = detector.anomaly_score(z_s)
                all_tpr.append(score >= detector.best_thr)
        
        TP = int(np.sum(all_tpr))
        FN = len(all_tpr) - TP
        FP = detector.FP_count
        TN = detector.TN_count
        Y_true = np.concatenate([np.ones(len(all_tpr)), np.zeros(len(detector.FP_DDD))])
        Y_pred = np.concatenate([np.array(all_tpr, dtype=int), np.array(detector.FP_DDD, dtype=int)])
        
        from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, matthews_corrcoef
        r = {
            'ang_str': ang_str, 'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
            'ACC': accuracy_score(Y_true, Y_pred),
            'F1': f1_score(Y_true, Y_pred, zero_division=0),
            'Recall': recall_score(Y_true, Y_pred, zero_division=0),
            'Precision': precision_score(Y_true, Y_pred, zero_division=0),
            'FPR': FP / len(detector.FP_DDD),
            'MCC': matthews_corrcoef(Y_true, Y_pred),
        }
        strength_results.append(r)
        print(f'  ang_str={ang_str:.2f} | ACC={r["ACC"]:.4f} F1={r["F1"]:.4f} Recall={r["Recall"]:.4f} MCC={r["MCC"]:.4f}')
    return strength_results

In [ ]:
# ---- Load Data & Evaluate ----
case_class = load_case()
z_noise_summary, v_est_summary = load_measurement()
load_load_pv()
test_dl_s, test_dl_u, valid_dl_s, valid_dl_u = load_dataset()

detector = Evaluation(case_class=case_class)
scaler_ = scaler()

# Select the threshold using TARGET_FPR.
detector.find_threshold(valid_dl_s, test_dl_s)

# Run attack strength sweep
strength_results = sweep_attack_strength(detector, test_dl_u, scaler_)

In [ ]:
# ---- Summary Table ----
import pandas as pd
df = pd.DataFrame(strength_results)
df = df[['ang_str', 'ACC', 'F1', 'Recall', 'Precision', 'MCC', 'FPR', 'TP', 'FP', 'TN', 'FN']]
print(f"Case: {sys_config['case_name']} | Model: Anomaly Transformer | Target FPR: {TARGET_FPR}")
print(df.round(4).to_string(index=False))

In [ ]:
# ---- Confusion Matrix (strongest attack) ----
from sklearn.metrics import confusion_matrix
import seaborn as sns

best_r = strength_results[-1]
cm_data = np.array([[best_r['TN'], best_r['FP']],
                     [best_r['FN'], best_r['TP']]])

plt.figure(figsize=(6, 5))
sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Normal', 'Pred Attack'],
            yticklabels=['True Normal', 'True Attack'])
plt.title(f'Confusion Matrix @ ang_str={best_r["ang_str"]:.2f}\n'
          f'ACC={best_r["ACC"]:.4f} F1={best_r["F1"]:.4f}')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Performance vs Attack Strength ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ACC / F1 / Recall vs ang_str
ax = axes[0]
ang = [r['ang_str'] for r in strength_results]
ax.plot(ang, [r['ACC'] for r in strength_results], 'o-', label='ACC', color='#2c3e50', linewidth=2)
ax.plot(ang, [r['F1'] for r in strength_results], 's-', label='F1', color='#e74c3c', linewidth=2)
ax.plot(ang, [r['Recall'] for r in strength_results], '^-', label='Recall', color='#27ae60', linewidth=2)
ax.set_xlabel('Attack Strength (ang_str)')
ax.set_ylabel('Score')
ax.set_title('Detection Performance vs Attack Strength')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# TP/FP vs ang_str
ax = axes[1]
ax.bar(np.array(ang) - 0.008, [r['TP'] for r in strength_results], width=0.015, label='TP', color='#27ae60', alpha=0.8)
ax.bar(np.array(ang) + 0.008, [r['FN'] for r in strength_results], width=0.015, label='FN', color='#e74c3c', alpha=0.8)
ax.set_xlabel('Attack Strength (ang_str)')
ax.set_ylabel('Count')
ax.set_title('TP / FN Counts')
ax.legend()
ax.grid(True, alpha=0.3)

# MCC vs ang_str
ax = axes[2]
ax.plot(ang, [r['MCC'] for r in strength_results], 'D-', color='#8e44ad', linewidth=2, markersize=8)
ax.set_xlabel('Attack Strength (ang_str)')
ax.set_ylabel('MCC')
ax.set_title('MCC vs Attack Strength')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.suptitle(f'Anomaly Transformer: {sys_config["case_name"]} (target FPR={TARGET_FPR})')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Save Results ----
import os
os.makedirs(f'metric/{sys_config["case_name"]}', exist_ok=True)
np.save(f'metric/{sys_config["case_name"]}/anomtrans_strength.npy', strength_results, allow_pickle=True)
print(f'Results saved to metric/{sys_config["case_name"]}/anomtrans_strength.npy')